# CHATBOT UNIVERSITARIO CON GRAMÁTICAS FORMALES

Este ejemplo implementa los conceptos vistos en clase:
1. Análisis Léxico (Tokenización)
2. Análisis Sintáctico (CFG)
3. Análisis Semántico (SDT)
4. Generación de Respuestas

In [1]:
import re
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from enum import Enum

In [2]:
# ============================================================================
# PASO 1: DEFINICIÓN DE ESTRUCTURAS DE DATOS
# ============================================================================

class TipoToken(Enum):
    """Enumera los tipos de tokens que reconocemos"""
    SALUDO = "SALUDO"
    DESPEDIDA = "DESPEDIDA"
    PREGUNTA_QUE = "PREGUNTA_QUE"
    PREGUNTA_COMO = "PREGUNTA_COMO"
    PREGUNTA_CUANDO = "PREGUNTA_CUANDO"
    QUIERO = "QUIERO"
    NECESITO = "NECESITO"
    INFORMACION = "INFORMACION"
    AYUDA = "AYUDA"
    SOBRE = "SOBRE"
    CURSO = "CURSO"
    CARRERA = "CARRERA"
    PRECIO = "PRECIO"
    HORARIO = "HORARIO"
    INSCRIPCION = "INSCRIPCION"
    REQUISITO = "REQUISITO"
    DESCONOCIDO = "DESCONOCIDO"

@dataclass
class Token:
    """Representa un token con su palabra original y tipo"""
    palabra: str
    tipo: TipoToken
    posicion: int

class TipoIntencion(Enum):
    """Tipos de intenciones que puede tener el usuario"""
    SALUDO = "SALUDO"
    DESPEDIDA = "DESPEDIDA"
    SOLICITUD_INFO = "SOLICITUD_INFO"
    PREGUNTA_DIRECTA = "PREGUNTA_DIRECTA"
    DESCONOCIDA = "DESCONOCIDA"

@dataclass
class Intencion:
    """Representa la intención parseada del usuario"""
    tipo: TipoIntencion
    tema: str = ""
    confianza: float = 1.0

In [3]:
# ============================================================================
# PASO 2: ANALIZADOR LÉXICO (TOKENIZADOR)
# ============================================================================

class AnalizadorLexico:
    """
    Implementa el análisis léxico: convierte texto en tokens
    """

    def __init__(self):
        # Diccionario léxico: mapea palabras a tipos de tokens
        self.lexico = {
            # Saludos
            'hola': TipoToken.SALUDO,
            'buenos': TipoToken.SALUDO,
            'buenas': TipoToken.SALUDO,
            'saludos': TipoToken.SALUDO,

            # Despedidas
            'adios': TipoToken.DESPEDIDA,
            'chau': TipoToken.DESPEDIDA,
            'hasta': TipoToken.DESPEDIDA,
            'gracias': TipoToken.DESPEDIDA,

            # Palabras interrogativas
            'que': TipoToken.PREGUNTA_QUE,
            'qué': TipoToken.PREGUNTA_QUE,
            'como': TipoToken.PREGUNTA_COMO,
            'cómo': TipoToken.PREGUNTA_COMO,
            'cuando': TipoToken.PREGUNTA_CUANDO,
            'cuándo': TipoToken.PREGUNTA_CUANDO,

            # Verbos de solicitud
            'quiero': TipoToken.QUIERO,
            'necesito': TipoToken.NECESITO,
            'busco': TipoToken.QUIERO,
            'me': TipoToken.QUIERO,

            # Sustantivos de solicitud
            'informacion': TipoToken.INFORMACION,
            'información': TipoToken.INFORMACION,
            'info': TipoToken.INFORMACION,
            'ayuda': TipoToken.AYUDA,
            'datos': TipoToken.INFORMACION,

            # Preposiciones
            'sobre': TipoToken.SOBRE,
            'de': TipoToken.SOBRE,
            'acerca': TipoToken.SOBRE,

            # Temas
            'curso': TipoToken.CURSO,
            'cursos': TipoToken.CURSO,
            'materia': TipoToken.CURSO,
            'materias': TipoToken.CURSO,
            'carrera': TipoToken.CARRERA,
            'carreras': TipoToken.CARRERA,
            'precio': TipoToken.PRECIO,
            'precios': TipoToken.PRECIO,
            'costo': TipoToken.PRECIO,
            'costos': TipoToken.PRECIO,
            'horario': TipoToken.HORARIO,
            'horarios': TipoToken.HORARIO,
            'inscripcion': TipoToken.INSCRIPCION,
            'inscripción': TipoToken.INSCRIPCION,
            'requisito': TipoToken.REQUISITO,
            'requisitos': TipoToken.REQUISITO,
        }

    def normalizar_texto(self, texto: str) -> str:
        """
        Normaliza el texto de entrada
        """
        # Convertir a minúsculas
        texto = texto.lower()

        # Quitar signos de puntuación
        texto = re.sub(r'[^\w\s]', '', texto)

        # Quitar espacios extra
        texto = re.sub(r'\s+', ' ', texto).strip()

        return texto

    def tokenizar(self, texto: str) -> List[Token]:
        """
        Convierte el texto en una lista de tokens
        """
        print(f"📝 ANÁLISIS LÉXICO")
        print(f"   Texto original: '{texto}'")

        # Normalizar
        texto_normalizado = self.normalizar_texto(texto)
        print(f"   Texto normalizado: '{texto_normalizado}'")

        # Dividir en palabras
        palabras = texto_normalizado.split()

        # Convertir palabras a tokens
        tokens = []
        for i, palabra in enumerate(palabras):
            tipo_token = self.lexico.get(palabra, TipoToken.DESCONOCIDO)
            token = Token(palabra, tipo_token, i)
            tokens.append(token)
            print(f"   '{palabra}' → {tipo_token.value}")

        print(f"   Tokens generados: {len(tokens)}")
        return tokens

In [4]:
# ============================================================================
# PASO 3: ANALIZADOR SINTÁCTICO (PARSER CFG)
# ============================================================================

class AnalizadorSintactico:
    """
    Implementa el análisis sintáctico usando una CFG simple
    """

    def __init__(self):
        # Definimos nuestra gramática como reglas de producción
        self.gramatica = {
            # S → SALUDO | DESPEDIDA | SOLICITUD | PREGUNTA
            'S': [
                ['SALUDO'],
                ['DESPEDIDA'],
                ['SOLICITUD'],
                ['PREGUNTA']
            ],

            # SALUDO → SALUDO_TOKEN
            'SALUDO': [
                ['SALUDO']
            ],

            # DESPEDIDA → DESPEDIDA_TOKEN
            'DESPEDIDA': [
                ['DESPEDIDA']
            ],

            # SOLICITUD → VERBO_SOLICITUD OBJETO_SOLICITUD
            # SOLICITUD → VERBO_SOLICITUD OBJETO_SOLICITUD SOBRE TEMA
            'SOLICITUD': [
                ['QUIERO', 'INFORMACION'],
                ['NECESITO', 'AYUDA'],
                ['QUIERO', 'INFORMACION', 'SOBRE', 'TEMA'],
                ['NECESITO', 'INFORMACION', 'SOBRE', 'TEMA']
            ],

            # PREGUNTA → PALABRA_INTERROGATIVA + resto
            'PREGUNTA': [
                ['PREGUNTA_QUE', 'TEMA'],
                ['PREGUNTA_COMO', 'TEMA'],
                ['PREGUNTA_CUANDO', 'TEMA']
            ],

            # TEMA → CURSO | CARRERA | PRECIO | HORARIO | etc.
            'TEMA': [
                ['CURSO'],
                ['CARRERA'],
                ['PRECIO'],
                ['HORARIO'],
                ['INSCRIPCION'],
                ['REQUISITO']
            ]
        }

    def parsear(self, tokens: List[Token]) -> Optional[Intencion]:
        """
        Analiza sintácticamente los tokens y determina la intención
        """
        print(f"\n🔍 ANÁLISIS SINTÁCTICO")
        print(f"   Tokens a analizar: {[t.tipo.value for t in tokens]}")

        # Intentamos hacer match con cada regla de la gramática
        for regla_nombre, producciones in self.gramatica.items():
            for produccion in producciones:
                if self._match_produccion(tokens, produccion):
                    intencion = self._crear_intencion(regla_nombre, tokens)
                    print(f"   ✅ Match encontrado: {regla_nombre} → {produccion}")
                    print(f"   Intención detectada: {intencion.tipo.value}")
                    return intencion

        print(f"   ❌ No se encontró match en la gramática")
        return Intencion(TipoIntencion.DESCONOCIDA)

    def _match_produccion(self, tokens: List[Token], produccion: List[str]) -> bool:
        """
        Verifica si los tokens hacen match con una producción específica
        """
        if len(tokens) < len(produccion):
            return False

        for i, simbolo in enumerate(produccion):
            if simbolo == 'TEMA':
                # TEMA es un no-terminal, verificamos si hay algún tema
                if not any(t.tipo.value in ['CURSO', 'CARRERA', 'PRECIO', 'HORARIO', 'INSCRIPCION', 'REQUISITO']
                          for t in tokens[i:]):
                    return False
            else:
                # Es un terminal, debe hacer match exacto
                if i >= len(tokens) or tokens[i].tipo.value != simbolo:
                    return False

        return True

    def _crear_intencion(self, regla: str, tokens: List[Token]) -> Intencion:
        """
        Crea una intención basada en la regla matched y los tokens
        """
        # Mapeo de reglas sintácticas a intenciones semánticas
        mapeo_intenciones = {
            'SALUDO': TipoIntencion.SALUDO,
            'DESPEDIDA': TipoIntencion.DESPEDIDA,
            'SOLICITUD': TipoIntencion.SOLICITUD_INFO,
            'PREGUNTA': TipoIntencion.PREGUNTA_DIRECTA
        }

        tipo_intencion = mapeo_intenciones.get(regla, TipoIntencion.DESCONOCIDA)

        # Extraer el tema si existe
        tema = ""
        temas_posibles = ['CURSO', 'CARRERA', 'PRECIO', 'HORARIO', 'INSCRIPCION', 'REQUISITO']
        for token in tokens:
            if token.tipo.value in temas_posibles:
                tema = token.tipo.value
                break

        return Intencion(tipo_intencion, tema)

In [5]:
# ============================================================================
# PASO 4: ANALIZADOR SEMÁNTICO Y GENERADOR DE RESPUESTAS
# ============================================================================

class AnalizadorSemantico:
    """
    Implementa el análisis semántico y la generación de respuestas
    """

    def __init__(self):
        # Base de conocimiento del chatbot
        self.base_conocimiento = {
            'CURSO': {
                'definicion': 'Los cursos son materias individuales que puedes tomar.',
                'ejemplos': ['Programación I', 'Matemática', 'Base de Datos'],
                'info_adicional': 'Ofrecemos cursos de programación, diseño, marketing y más.'
            },
            'CARRERA': {
                'definicion': 'Las carreras son programas completos de estudio.',
                'ejemplos': ['Ingeniería en Sistemas', 'Diseño Gráfico', 'Marketing'],
                'info_adicional': 'Todas nuestras carreras tienen validez oficial.'
            },
            'PRECIO': {
                'definicion': 'Los precios varían según el curso o carrera.',
                'ejemplos': ['Cursos: $5000-15000', 'Carreras: $20000-50000'],
                'info_adicional': 'Ofrecemos planes de pago y becas.'
            },
            'HORARIO': {
                'definicion': 'Tenemos horarios flexibles para adaptarse a tu rutina.',
                'ejemplos': ['Mañana: 8-12hs', 'Tarde: 14-18hs', 'Noche: 19-22hs'],
                'info_adicional': 'También ofrecemos modalidad online.'
            },
            'INSCRIPCION': {
                'definicion': 'El proceso de inscripción es simple y rápido.',
                'ejemplos': ['Online', 'Presencial', 'Por teléfono'],
                'info_adicional': 'Solo necesitas DNI y certificado de estudios.'
            },
            'REQUISITO': {
                'definicion': 'Los requisitos varían según el curso o carrera.',
                'ejemplos': ['Secundario completo', 'Conocimientos básicos', 'Sin requisitos'],
                'info_adicional': 'Para algunos cursos avanzados se requiere experiencia previa.'
            }
        }

        # Plantillas de respuestas
        self.plantillas_respuestas = {
            TipoIntencion.SALUDO: [
                "¡Hola! Soy tu asistente virtual de la Universidad. ¿En qué puedo ayudarte?",
                "¡Buenos días! Estoy aquí para ayudarte con información académica.",
                "¡Saludos! ¿Qué información necesitas sobre nuestros cursos y carreras?"
            ],
            TipoIntencion.DESPEDIDA: [
                "¡Hasta luego! Espero haber sido de ayuda.",
                "¡Que tengas un excelente día! No dudes en consultarme cuando lo necesites.",
                "¡Adiós! Estoy aquí cuando necesites más información."
            ],
            TipoIntencion.DESCONOCIDA: [
                "Lo siento, no entendí tu consulta. ¿Podrías reformularla?",
                "No estoy seguro de qué necesitas. ¿Podrías ser más específico?",
                "Disculpa, no comprendí. Puedes preguntarme sobre cursos, carreras, precios, horarios, inscripciones o requisitos."
            ]
        }

    def procesar_intencion(self, intencion: Intencion) -> str:
        """
        Procesa la intención y genera una respuesta apropiada
        """
        print(f"\n🧠 ANÁLISIS SEMÁNTICO")
        print(f"   Procesando intención: {intencion.tipo.value}")
        if intencion.tema:
            print(f"   Tema detectado: {intencion.tema}")

        # Generar respuesta según el tipo de intención
        if intencion.tipo == TipoIntencion.SALUDO:
            return self._generar_saludo()

        elif intencion.tipo == TipoIntencion.DESPEDIDA:
            return self._generar_despedida()

        elif intencion.tipo == TipoIntencion.SOLICITUD_INFO:
            return self._generar_info(intencion.tema)

        elif intencion.tipo == TipoIntencion.PREGUNTA_DIRECTA:
            return self._generar_respuesta_pregunta(intencion.tema)

        else:
            return self._generar_desconocida()

    def _generar_saludo(self) -> str:
        import random
        return random.choice(self.plantillas_respuestas[TipoIntencion.SALUDO])

    def _generar_despedida(self) -> str:
        import random
        return random.choice(self.plantillas_respuestas[TipoIntencion.DESPEDIDA])

    def _generar_info(self, tema: str) -> str:
        """Genera información sobre un tema específico"""
        if tema in self.base_conocimiento:
            info = self.base_conocimiento[tema]
            respuesta = f"📚 Información sobre {tema.lower()}s:\n\n"
            respuesta += f"• {info['definicion']}\n"
            respuesta += f"• Ejemplos: {', '.join(info['ejemplos'])}\n"
            respuesta += f"• {info['info_adicional']}\n\n"
            respuesta += "¿Necesitas información más específica sobre algún aspecto?"
            return respuesta
        else:
            return "Puedo ayudarte con información sobre cursos, carreras, precios, horarios, inscripciones y requisitos. ¿Sobre cuál te interesa saber?"

    def _generar_respuesta_pregunta(self, tema: str) -> str:
        """Genera respuesta a una pregunta directa"""
        return self._generar_info(tema)  # Por simplicidad, usamos la misma lógica

    def _generar_desconocida(self) -> str:
        import random
        return random.choice(self.plantillas_respuestas[TipoIntencion.DESCONOCIDA])

In [6]:
# ============================================================================
# PASO 5: CHATBOT PRINCIPAL
# ============================================================================

class ChatbotUniversitario:
    """
    Clase principal que integra todos los componentes
    """

    def __init__(self):
        self.analizador_lexico = AnalizadorLexico()
        self.analizador_sintactico = AnalizadorSintactico()
        self.analizador_semantico = AnalizadorSemantico()
        self.historial_conversacion = []

    def procesar_mensaje(self, mensaje: str) -> str:
        """
        Procesa un mensaje del usuario y genera una respuesta
        """
        print("=" * 60)
        print(f"🤖 PROCESANDO MENSAJE: '{mensaje}'")
        print("=" * 60)

        # Paso 1: Análisis Léxico
        tokens = self.analizador_lexico.tokenizar(mensaje)

        # Paso 2: Análisis Sintáctico
        intencion = self.analizador_sintactico.parsear(tokens)

        # Paso 3: Análisis Semántico y Generación de Respuesta
        respuesta = self.analizador_semantico.procesar_intencion(intencion)

        # Guardar en historial
        self.historial_conversacion.append({
            'mensaje': mensaje,
            'tokens': tokens,
            'intencion': intencion,
            'respuesta': respuesta
        })

        print(f"\n💬 RESPUESTA GENERADA:")
        print(f"   {respuesta}")
        print("=" * 60)

        return respuesta

    def mostrar_estadisticas(self):
        """Muestra estadísticas de la conversación"""
        print(f"\n📊 ESTADÍSTICAS DE LA CONVERSACIÓN:")
        print(f"   Total de mensajes procesados: {len(self.historial_conversacion)}")

        # Contar tipos de intenciones
        contador_intenciones = {}
        for entrada in self.historial_conversacion:
            tipo = entrada['intencion'].tipo.value
            contador_intenciones[tipo] = contador_intenciones.get(tipo, 0) + 1

        print(f"   Distribución de intenciones:")
        for tipo, cantidad in contador_intenciones.items():
            print(f"     - {tipo}: {cantidad}")

In [11]:
# ============================================================================
# PASO 6: DEMOSTRACIÓN Y PRUEBAS
# ============================================================================

def main():
    """
    Función principal para demostrar el chatbot
    """
    print("🎓 CHATBOT UNIVERSITARIO CON GRAMÁTICAS FORMALES")
    print("=" * 60)
    print("Este chatbot demuestra los conceptos de:")
    print("• Análisis Léxico (Tokenización)")
    print("• Análisis Sintáctico (CFG)")
    print("• Análisis Semántico (SDT)")
    print("=" * 60)

    # Crear el chatbot
    chatbot = ChatbotUniversitario()

    # Mensajes de prueba
    mensajes_prueba = [
        "Hola, buenos días",
        "Quiero información sobre cursos",
        "Necesito ayuda sobre precios",
        "Qué horarios tienen",
        "Como son los requisitos",
        "Cuando puedo inscribirme",
        "Gracias, adiós",
        "xyz abc def"  # Mensaje que no debería ser reconocido
    ]

    print(f"\n🧪 EJECUTANDO PRUEBAS AUTOMÁTICAS:")
    print("-" * 60)

    for i, mensaje in enumerate(mensajes_prueba, 1):
        print(f"\n📨 PRUEBA {i}/{len(mensajes_prueba)}")
        respuesta = chatbot.procesar_mensaje(mensaje)
        print(f"\n👤 Usuario: {mensaje}")
        print(f"🤖 Bot: {respuesta}")

        if i < len(mensajes_prueba):
            input("\n⏸️  Presiona ENTER para continuar con la siguiente prueba...")

    # Mostrar estadísticas
    chatbot.mostrar_estadisticas()

    # Modo interactivo
    print(f"\n🎮 MODO INTERACTIVO (escribe 'salir' para terminar):")
    print("-" * 60)

    while True:
        try:
            mensaje_usuario = input("\n👤 Tú: ").strip()

            if mensaje_usuario.lower() in ['salir', 'exit', 'quit']:
                print("🤖 Bot: ¡Hasta luego! Gracias por probar el chatbot.")
                break

            if mensaje_usuario:
                respuesta = chatbot.procesar_mensaje(mensaje_usuario)
                print(f"🤖 Bot: {respuesta}")

        except KeyboardInterrupt:
            print("\n\n🤖 Bot: ¡Hasta luego! Gracias por probar el chatbot.")
            break
        except Exception as e:
            print(f"❌ Error: {e}")

if __name__ == "__main__":
    main()

🎓 CHATBOT UNIVERSITARIO CON GRAMÁTICAS FORMALES
Este chatbot demuestra los conceptos de:
• Análisis Léxico (Tokenización)
• Análisis Sintáctico (CFG)
• Análisis Semántico (SDT)

🧪 EJECUTANDO PRUEBAS AUTOMÁTICAS:
------------------------------------------------------------

📨 PRUEBA 1/8
🤖 PROCESANDO MENSAJE: 'Hola, buenos días'
📝 ANÁLISIS LÉXICO
   Texto original: 'Hola, buenos días'
   Texto normalizado: 'hola buenos días'
   'hola' → SALUDO
   'buenos' → SALUDO
   'días' → DESCONOCIDO
   Tokens generados: 3

🔍 ANÁLISIS SINTÁCTICO
   Tokens a analizar: ['SALUDO', 'SALUDO', 'DESCONOCIDO']
   ✅ Match encontrado: S → ['SALUDO']
   Intención detectada: DESCONOCIDA

🧠 ANÁLISIS SEMÁNTICO
   Procesando intención: DESCONOCIDA

💬 RESPUESTA GENERADA:
   Disculpa, no comprendí. Puedes preguntarme sobre cursos, carreras, precios, horarios, inscripciones o requisitos.

👤 Usuario: Hola, buenos días
🤖 Bot: Disculpa, no comprendí. Puedes preguntarme sobre cursos, carreras, precios, horarios, inscripci

# Consignas

#Consigna 1: Implementar preguntas con múltiples palabras interrogativas

Modifica la gramática para que el chatbot entienda preguntas más complejas con múltiples palabras interrogativas.
Tareas específicas:

Agrega estas nuevas reglas a la gramática:

  PREGUNTA_COMPLEJA → PREGUNTA_DONDE VERBO TEMA

  PREGUNTA_COMPLEJA → PREGUNTA_CUANTO VERBO TEMA  
  
  PREGUNTA_MULTIPLE → PREGUNTA_QUE TEMA VERBO PREGUNTA_CUANDO

Crea nuevos TipoToken: DONDE, CUANTO, VERBO

Agrega palabras: "donde", "dónde", "cuanto", "cuánto", "cuesta", "queda", "empieza"

Prueba con:

"Dónde queda la universidad"

"Cuánto cuesta el curso"

"Qué horarios hay cuando empiezan"

Entregable: Gramática extendida + método de parsing modificado + ejemplos funcionando + diagrama de la nueva estructura gramatical.

In [12]:

class TipoToken(Enum):
    """Enumera los tipos de tokens que reconocemos"""
    SALUDO = "SALUDO"
    DESPEDIDA = "DESPEDIDA"
    PREGUNTA_QUE = "PREGUNTA_QUE"
    PREGUNTA_COMO = "PREGUNTA_COMO"
    PREGUNTA_CUANDO = "PREGUNTA_CUANDO"
    DONDE = "DONDE"            # nuevo
    CUANTO = "CUANTO"          # nuevo
    QUIERO = "QUIERO"
    NECESITO = "NECESITO"
    INFORMACION = "INFORMACION"
    AYUDA = "AYUDA"
    SOBRE = "SOBRE"
    CURSO = "CURSO"
    CARRERA = "CARRERA"
    PRECIO = "PRECIO"
    HORARIO = "HORARIO"
    INSCRIPCION = "INSCRIPCION"
    REQUISITO = "REQUISITO"
    VERBO = "VERBO"            # nuevo
    DESCONOCIDO = "DESCONOCIDO"

@dataclass
class Token:
    palabra: str
    tipo: TipoToken
    posicion: int

class TipoIntencion(Enum):
    SALUDO = "SALUDO"
    DESPEDIDA = "DESPEDIDA"
    SOLICITUD_INFO = "SOLICITUD_INFO"
    PREGUNTA_DIRECTA = "PREGUNTA_DIRECTA"
    DESCONOCIDA = "DESCONOCIDA"

@dataclass
class Intencion:
    tipo: TipoIntencion
    tema: str = ""
    confianza: float = 1.0

class AnalizadorLexico:
    def __init__(self):
        self.lexico = {
            # Saludos
            'hola': TipoToken.SALUDO,
            'buenos': TipoToken.SALUDO,
            'buenas': TipoToken.SALUDO,
            'saludos': TipoToken.SALUDO,

            # Despedidas
            'adios': TipoToken.DESPEDIDA,
            'chau': TipoToken.DESPEDIDA,
            'hasta': TipoToken.DESPEDIDA,
            'gracias': TipoToken.DESPEDIDA,

            # Palabras interrogativas
            'que': TipoToken.PREGUNTA_QUE,
            'qué': TipoToken.PREGUNTA_QUE,
            'como': TipoToken.PREGUNTA_COMO,
            'cómo': TipoToken.PREGUNTA_COMO,
            'cuando': TipoToken.PREGUNTA_CUANDO,
            'cuándo': TipoToken.PREGUNTA_CUANDO,

            # Nuevas interrogativas solicitadas
            'donde': TipoToken.DONDE,
            'dónde': TipoToken.DONDE,
            'cuanto': TipoToken.CUANTO,
            'cuánto': TipoToken.CUANTO,

            # Verbos sencillos (se consideran VERBO en la gramática)
            'cuesta': TipoToken.VERBO,
            'queda': TipoToken.VERBO,
            'empieza': TipoToken.VERBO,
            'empiezan': TipoToken.VERBO,
            'quedan': TipoToken.VERBO,
            'cuestan': TipoToken.VERBO,

            # Verbos de solicitud
            'quiero': TipoToken.QUIERO,
            'necesito': TipoToken.NECESITO,
            'busco': TipoToken.QUIERO,
            'me': TipoToken.QUIERO,

            # Sustantivos de solicitud
            'informacion': TipoToken.INFORMACION,
            'información': TipoToken.INFORMACION,
            'info': TipoToken.INFORMACION,
            'ayuda': TipoToken.AYUDA,
            'datos': TipoToken.INFORMACION,

            # Preposiciones
            'sobre': TipoToken.SOBRE,
            'de': TipoToken.SOBRE,
            'acerca': TipoToken.SOBRE,

            # Temas
            'curso': TipoToken.CURSO,
            'cursos': TipoToken.CURSO,
            'materia': TipoToken.CURSO,
            'materias': TipoToken.CURSO,
            'carrera': TipoToken.CARRERA,
            'carreras': TipoToken.CARRERA,
            'precio': TipoToken.PRECIO,
            'precios': TipoToken.PRECIO,
            'costo': TipoToken.PRECIO,
            'costos': TipoToken.PRECIO,
            'horario': TipoToken.HORARIO,
            'horarios': TipoToken.HORARIO,
            'inscripcion': TipoToken.INSCRIPCION,
            'inscripción': TipoToken.INSCRIPCION,
            'requisito': TipoToken.REQUISITO,
            'requisitos': TipoToken.REQUISITO,

            # ejemplo: 'universidad' lo puedo mapear a CARRERA si quieres:
            'universidad': TipoToken.CARRERA,
        }

    def normalizar_texto(self, texto: str) -> str:
        texto = texto.lower()
        texto = re.sub(r'[^\w\sáéíóúüñÁÉÍÓÚÜÑ]', '', texto)  # mantener acentos y ñ
        texto = re.sub(r'\s+', ' ', texto).strip()
        return texto

    def tokenizar(self, texto: str) -> List[Token]:
        print(f"📝 ANÁLISIS LÉXICO")
        print(f"   Texto original: '{texto}'")
        texto_normalizado = self.normalizar_texto(texto)
        print(f"   Texto normalizado: '{texto_normalizado}'")
        palabras = texto_normalizado.split()
        tokens = []
        for i, palabra in enumerate(palabras):
            tipo_token = self.lexico.get(palabra, TipoToken.DESCONOCIDO)
            token = Token(palabra, tipo_token, i)
            tokens.append(token)
            print(f"   '{palabra}' → {tipo_token.value}")
        print(f"   Tokens generados: {len(tokens)}")
        return tokens

class AnalizadorSintactico:
    def __init__(self):
        self.gramatica = {
            # S → SALUDO | DESPEDIDA | SOLICITUD | PREGUNTA | PREGUNTA_COMPLEJA | PREGUNTA_MULTIPLE
            'S': [
                ['SALUDO'],
                ['DESPEDIDA'],
                ['SOLICITUD'],
                ['PREGUNTA'],
                ['PREGUNTA_COMPLEJA'],
                ['PREGUNTA_MULTIPLE']
            ],

            # SALUDO → SALUDO
            'SALUDO': [['SALUDO']],

            # DESPEDIDA → DESPEDIDA
            'DESPEDIDA': [['DESPEDIDA']],

            # SOLICITUD → QUIERO INFORMACION | NECESITO AYUDA | QUIERO INFORMACION SOBRE TEMA ...
            'SOLICITUD': [
                ['QUIERO', 'INFORMACION'],
                ['NECESITO', 'AYUDA'],
                ['QUIERO', 'INFORMACION', 'SOBRE', 'TEMA'],
                ['NECESITO', 'INFORMACION', 'SOBRE', 'TEMA']
            ],

            # PREGUNTA simple
            'PREGUNTA': [
                ['PREGUNTA_QUE', 'TEMA'],
                ['PREGUNTA_COMO', 'TEMA'],
                ['PREGUNTA_CUANDO', 'TEMA']
            ],

            # PREGUNTA_COMPLEJA → DONDE VERBO TEMA
            # PREGUNTA_COMPLEJA → CUANTO VERBO TEMA
            'PREGUNTA_COMPLEJA': [
                ['DONDE', 'VERBO', 'TEMA'],
                ['CUANTO', 'VERBO', 'TEMA']
            ],

            # PREGUNTA_MULTIPLE → PREGUNTA_QUE TEMA VERBO PREGUNTA_CUANDO
            'PREGUNTA_MULTIPLE': [
                ['PREGUNTA_QUE', 'TEMA', 'VERBO', 'PREGUNTA_CUANDO']
            ],

            # TEMA → CURSO | CARRERA | PRECIO | HORARIO | INSCRIPCION | REQUISITO
            'TEMA': [
                ['CURSO'],
                ['CARRERA'],
                ['PRECIO'],
                ['HORARIO'],
                ['INSCRIPCION'],
                ['REQUISITO']
            ]
        }

    def parsear(self, tokens: List[Token]) -> Optional[Intencion]:
        print(f"\n🔍 ANÁLISIS SINTÁCTICO")
        print(f"   Tokens a analizar: {[t.tipo.value for t in tokens]}")

        for regla_nombre, producciones in self.gramatica.items():
            for produccion in producciones:
                if self._match_produccion(tokens, produccion):
                    intencion = self._crear_intencion(regla_nombre, tokens)
                    print(f"   ✅ Match encontrado: {regla_nombre} → {produccion}")
                    print(f"   Intención detectada: {intencion.tipo.value}")
                    return intencion

        print(f"   ❌ No se encontró match en la gramática")
        return Intencion(TipoIntencion.DESCONOCIDA)

    def _match_produccion(self, tokens: List[Token], produccion: List[str]) -> bool:
        """
        Matching más flexible: intentamos alinear la producción en cualquier
        posición de la lista de tokens (sliding window). TEMA es un no-terminal
        que se satisface si hay algún token de tipo tema en la ventana esperada.
        """
        n = len(tokens)
        m = len(produccion)
        if m == 0:
            return False

        # intentamos cada posición de inicio posible
        for start in range(0, n):
            # si la producción excede longitud, continue
            if start + m > n:
                continue

            ok = True
            for j, simbolo in enumerate(produccion):
                token = tokens[start + j]
                if simbolo == 'TEMA':
                    if token.tipo.value not in ['CURSO', 'CARRERA', 'PRECIO', 'HORARIO', 'INSCRIPCION', 'REQUISITO']:
                        ok = False
                        break
                else:
                    # terminal exacto
                    # permitimos que PREGUNTA_QUE y similares vengan de tokens PREGUNTA_QUE, PREGUNTA_COMO, etc.
                    if token.tipo.value != simbolo:
                        ok = False
                        break
            if ok:
                return True

        # También permitimos que TEMA aparezca más adelante (por ejemplo: DONDE VERBO ... la universidad)
        # Intent: para producciones que terminan en TEMA, buscaremos TEMA en tokens posterior.
        # (ej: ['DONDE','VERBO','TEMA'] y tokens = DONDE, VERBO, 'la', 'universidad' → universidad mapeada a CARRERA)
        # Reintento para producciones que terminan con TEMA y no coincidieron por exactitud:
        if produccion and produccion[-1] == 'TEMA':
            # buscamos indices i,j tal que primeros m-1 coinciden contiguos y después haya algún token tema
            pref_len = len(produccion) - 1
            for start in range(0, n):
                if start + pref_len > n:
                    continue
                # comprobar prefijos
                ok_pref = True
                for j in range(pref_len):
                    simbolo = produccion[j]
                    token = tokens[start + j]
                    if token.tipo.value != simbolo:
                        ok_pref = False
                        break
                if not ok_pref:
                    continue
                # buscar tema en los siguientes tokens
                for k in range(start + pref_len, n):
                    if tokens[k].tipo.value in ['CURSO', 'CARRERA', 'PRECIO', 'HORARIO', 'INSCRIPCION', 'REQUISITO']:
                        return True
        return False

    def _crear_intencion(self, regla: str, tokens: List[Token]) -> Intencion:
        mapeo_intenciones = {
            'SALUDO': TipoIntencion.SALUDO,
            'DESPEDIDA': TipoIntencion.DESPEDIDA,
            'SOLICITUD': TipoIntencion.SOLICITUD_INFO,
            'PREGUNTA': TipoIntencion.PREGUNTA_DIRECTA,
            'PREGUNTA_COMPLEJA': TipoIntencion.PREGUNTA_DIRECTA,
            'PREGUNTA_MULTIPLE': TipoIntencion.PREGUNTA_DIRECTA
        }
        tipo_intencion = mapeo_intenciones.get(regla, TipoIntencion.DESCONOCIDA)

        # Extraer el tema si existe (buscamos el primer token que sea tema)
        tema = ""
        temas_posibles = ['CURSO', 'CARRERA', 'PRECIO', 'HORARIO', 'INSCRIPCION', 'REQUISITO']
        for token in tokens:
            if token.tipo.value in temas_posibles:
                tema = token.tipo.value
                break

        return Intencion(tipo_intencion, tema)




In [14]:
# ============================================================================
# PASO 6: DEMOSTRACIÓN Y PRUEBAS
# ============================================================================

def main():
    """
    Función principal para demostrar el chatbot
    """
    print("🎓 CHATBOT UNIVERSITARIO CON GRAMÁTICAS FORMALES")
    print("=" * 60)
    print("Este chatbot demuestra los conceptos de:")
    print("• Análisis Léxico (Tokenización)")
    print("• Análisis Sintáctico (CFG)")
    print("• Análisis Semántico (SDT)")
    print("=" * 60)

    # Crear el chatbot
    chatbot = ChatbotUniversitario()

    # Mensajes de prueba
    mensajes_prueba = [
        "Hola, buenos días",
        "Quiero información sobre cursos",
        "Necesito ayuda sobre precios",
        "Qué horarios tienen",
        "Como son los requisitos",
        "Cuando puedo inscribirme",
        "Gracias, adiós",
        "xyz abc def"  # Mensaje que no debería ser reconocido
    ]

    print(f"\n🧪 EJECUTANDO PRUEBAS AUTOMÁTICAS:")
    print("-" * 60)

    for i, mensaje in enumerate(mensajes_prueba, 1):
        print(f"\n📨 PRUEBA {i}/{len(mensajes_prueba)}")
        respuesta = chatbot.procesar_mensaje(mensaje)
        print(f"\n👤 Usuario: {mensaje}")
        print(f"🤖 Bot: {respuesta}")

        if i < len(mensajes_prueba):
            input("\n⏸️  Presiona ENTER para continuar con la siguiente prueba...")

    # Mostrar estadísticas
    chatbot.mostrar_estadisticas()

    # Modo interactivo
    print(f"\n🎮 MODO INTERACTIVO (escribe 'salir' para terminar):")
    print("-" * 60)

    while True:
        try:
            mensaje_usuario = input("\n👤 Tú: ").strip()

            if mensaje_usuario.lower() in ['salir', 'exit', 'quit']:
                print("🤖 Bot: ¡Hasta luego! Gracias por probar el chatbot.")
                break

            if mensaje_usuario:
                respuesta = chatbot.procesar_mensaje(mensaje_usuario)
                print(f"🤖 Bot: {respuesta}")

        except KeyboardInterrupt:
            print("\n\n🤖 Bot: ¡Hasta luego! Gracias por probar el chatbot.")
            break
        except Exception as e:
            print(f"❌ Error: {e}")

if __name__ == "__main__":
    main()

🎓 CHATBOT UNIVERSITARIO CON GRAMÁTICAS FORMALES
Este chatbot demuestra los conceptos de:
• Análisis Léxico (Tokenización)
• Análisis Sintáctico (CFG)
• Análisis Semántico (SDT)

🧪 EJECUTANDO PRUEBAS AUTOMÁTICAS:
------------------------------------------------------------

📨 PRUEBA 1/8
🤖 PROCESANDO MENSAJE: 'Hola, buenos días'
📝 ANÁLISIS LÉXICO
   Texto original: 'Hola, buenos días'
   Texto normalizado: 'hola buenos días'
   'hola' → SALUDO
   'buenos' → SALUDO
   'días' → DESCONOCIDO
   Tokens generados: 3

🔍 ANÁLISIS SINTÁCTICO
   Tokens a analizar: ['SALUDO', 'SALUDO', 'DESCONOCIDO']
   ✅ Match encontrado: S → ['SALUDO']
   Intención detectada: DESCONOCIDA

🧠 ANÁLISIS SEMÁNTICO
   Procesando intención: DESCONOCIDA

💬 RESPUESTA GENERADA:
   Lo siento, no entendí tu consulta. ¿Podrías reformularla?

👤 Usuario: Hola, buenos días
🤖 Bot: Lo siento, no entendí tu consulta. ¿Podrías reformularla?

📨 PRUEBA 2/8
🤖 PROCESANDO MENSAJE: 'Quiero información sobre cursos'
📝 ANÁLISIS LÉXICO
   Te

#Consigna 2: Agregar soporte para negaciones y condiciones

Implementa reglas gramaticales para manejar oraciones negativas y condicionales.
Tareas específicas:

Agrega TipoToken.NEGACION para: "no", "nunca", "tampoco"

Agrega TipoToken.CONDICIONAL para: "si", "cuando", "mientras"

Crea estas reglas gramaticales:

  PREGUNTA_NEGATIVA → NEGACION VERBO TEMA

  SOLICITUD_CONDICIONAL → CONDICIONAL CONDICION ENTONCES SOLICITUD

Modifica el analizador sintáctico para manejar estos casos

Prueba con:

"No tengo información sobre precios"

"Si no hay cupo, qué alternativas hay"

"Cuándo no hay clases"

Entregable: Implementación completa + casos de prueba + explicación de cómo las negaciones cambian el significado semántico.

In [13]:
from enum import Enum, auto
from dataclasses import dataclass
class TipoToken(Enum):
    VERBO = auto()
    TEMA = auto()
    PREGUNTA = auto()
    SOLICITUD = auto()
    NEGACION = auto()       # "no", "nunca", "tampoco"
    CONDICIONAL = auto()    # "si", "cuando", "mientras"
    ENTONCES = auto()       # "entonces"

@dataclass
class Token:
    tipo: TipoToken
    valor: str

# ================================
# 2. Lexer
# ================================
palabras_reservadas = {
    # En AnalizadorLexico.__init__(), dentro de self.lexico:
    'hay': TipoToken.VERBO,
    'tengo': TipoToken.VERBO,
    'queda': TipoToken.VERBO,
    'cuesta': TipoToken.VERBO,
    'empieza': TipoToken.VERBO,
    "no": TipoToken.NEGACION,
    "nunca": TipoToken.NEGACION,
    "tampoco": TipoToken.NEGACION,
    "si": TipoToken.CONDICIONAL,
    "cuando": TipoToken.CONDICIONAL,
    "mientras": TipoToken.CONDICIONAL,
    "entonces": TipoToken.ENTONCES,
    "qué": TipoToken.PREGUNTA,
    "alternativas": TipoToken.TEMA,
    "precios": TipoToken.TEMA,
    "información": TipoToken.TEMA,
    "cupo": TipoToken.TEMA,
    "clases": TipoToken.TEMA,
    "tengo": TipoToken.VERBO,
    "hay": TipoToken.VERBO,
}

def lexer(oracion: str):
    tokens = []
    for palabra in oracion.lower().split():
        if palabra in palabras_reservadas:
            tokens.append(Token(palabras_reservadas[palabra], palabra))
        else:
            # por defecto: asumimos tema
            tokens.append(Token(TipoToken.TEMA, palabra))
    return tokens

# ================================
# 3. Parser
# ================================
class Parser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def match(self, tipo):
        if self.pos < len(self.tokens) and self.tokens[self.pos].tipo == tipo:
            tok = self.tokens[self.pos]
            self.pos += 1
            return tok
        raise SyntaxError(f"Se esperaba {tipo}, encontrado {self.tokens[self.pos].tipo}")

    def parse(self):
        # Intentamos primero reglas nuevas
        if self.pos < len(self.tokens) and self.tokens[self.pos].tipo == TipoToken.NEGACION:
            return self.pregunta_negativa()
        if self.pos < len(self.tokens) and self.tokens[self.pos].tipo == TipoToken.CONDICIONAL:
            return self.solicitud_condicional()
        # fallback: enunciado simple
        return ("ENUNCIADO_GENERAL", [t.valor for t in self.tokens])

    def pregunta_negativa(self):
        neg = self.match(TipoToken.NEGACION)
        verbo = self.match(TipoToken.VERBO)
        tema = self.match(TipoToken.TEMA)
        return ("PREGUNTA_NEGATIVA", neg.valor, verbo.valor, tema.valor)

    def solicitud_condicional(self):
        cond = self.match(TipoToken.CONDICIONAL)
        condicion = self.condicion()
        solicitud = None
        if self.pos < len(self.tokens) and self.tokens[self.pos].tipo == TipoToken.ENTONCES:
            self.match(TipoToken.ENTONCES)
            solicitud = self.solicitud()
        return ("SOLICITUD_CONDICIONAL", cond.valor, condicion, solicitud)

    def condicion(self):
        if self.pos < len(self.tokens) and self.tokens[self.pos].tipo == TipoToken.NEGACION:
            return self.pregunta_negativa()
        else:
            verbo = self.match(TipoToken.VERBO)
            tema = self.match(TipoToken.TEMA)
            return ("CONDICION_SIMPLE", verbo.valor, tema.valor)

    def solicitud(self):
        # Soporta "qué" antes del verbo/tema
        if self.pos < len(self.tokens) and self.tokens[self.pos].tipo == TipoToken.PREGUNTA:
            self.match(TipoToken.PREGUNTA)

        # Caso 1: verbo seguido de tema
        if self.pos < len(self.tokens) and self.tokens[self.pos].tipo == TipoToken.VERBO:
            verbo = self.match(TipoToken.VERBO)
            if self.pos < len(self.tokens) and self.tokens[self.pos].tipo == TipoToken.TEMA:
                tema = self.match(TipoToken.TEMA)
                return ("SOLICITUD", verbo.valor, tema.valor)
            return ("SOLICITUD", verbo.valor, None)

        # Caso 2: tema primero (ej. "qué alternativas hay")
        elif self.pos < len(self.tokens) and self.tokens[self.pos].tipo == TipoToken.TEMA:
            tema = self.match(TipoToken.TEMA)
            # si luego hay un verbo, lo tomamos
            if self.pos < len(self.tokens) and self.tokens[self.pos].tipo == TipoToken.VERBO:
                verbo = self.match(TipoToken.VERBO)
                return ("SOLICITUD", verbo.valor, tema.valor)
            return ("SOLICITUD", None, tema.valor)

        else:
            raise SyntaxError(f"Se esperaba VERBO o TEMA, encontrado {self.tokens[self.pos].tipo}")


# ================================
# 4. Casos de prueba
# ================================
tests = [
    "no tengo información sobre precios",
    "si no hay cupo entonces qué alternativas hay",
    "cuando no hay clases"
]

for t in tests:
    print("\nOración:", t)
    toks = lexer(t)
    print("Tokens:", [(tk.tipo.name, tk.valor) for tk in toks])
    parser = Parser(toks)
    arbol = parser.parse()
    print("AST:", arbol)




Oración: no tengo información sobre precios
Tokens: [('NEGACION', 'no'), ('VERBO', 'tengo'), ('TEMA', 'información'), ('TEMA', 'sobre'), ('TEMA', 'precios')]
AST: ('PREGUNTA_NEGATIVA', 'no', 'tengo', 'información')

Oración: si no hay cupo entonces qué alternativas hay
Tokens: [('CONDICIONAL', 'si'), ('NEGACION', 'no'), ('VERBO', 'hay'), ('TEMA', 'cupo'), ('ENTONCES', 'entonces'), ('PREGUNTA', 'qué'), ('TEMA', 'alternativas'), ('VERBO', 'hay')]
AST: ('SOLICITUD_CONDICIONAL', 'si', ('PREGUNTA_NEGATIVA', 'no', 'hay', 'cupo'), ('SOLICITUD', 'hay', 'alternativas'))

Oración: cuando no hay clases
Tokens: [('CONDICIONAL', 'cuando'), ('NEGACION', 'no'), ('VERBO', 'hay'), ('TEMA', 'clases')]
AST: ('SOLICITUD_CONDICIONAL', 'cuando', ('PREGUNTA_NEGATIVA', 'no', 'hay', 'clases'), None)


La implementacion de preguntas negativas complejiza el codigo en alta medida. Hay varias formas verbales de negar una pregunta, eso hace compleja la interpretacion de la misma.